In [4]:
!pip install -r requirements.txt

## Random Forest Classifier

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score,accuracy_score

train_df = pd.read_csv('./data/training_dataset.csv')
val_df = pd.read_csv('./data/validation_dataset.csv')

# Fix time column type
train_df['time'] = pd.to_datetime(train_df['time'])
val_df['time'] = pd.to_datetime(val_df['time'])

# Extract features/targets
def prepare_X_y(df_part):
    X = df_part[['time', 'lat', 'lon', 'lat_rad', 'lon_rad', 'sin_lat', 'cos_lat', 'sin_lon', 'cos_lon']].copy()

    # Add new features
    X['month'] = pd.to_datetime(df_part['time']).dt.month
    X['season'] = pd.to_datetime(df_part['time']).dt.month % 12 // 3
    X['lat_lon_interaction'] = X['lat'] * X['lon']
    
    X['time'] = X['time'].dt.strftime('%Y%m%d').astype(int)
    
    y = df_part['sla'].copy()
    return X, y

X_train, y_train = prepare_X_y(train_df)
X_val, y_val = prepare_X_y(val_df)

# Ensure consistent feature names by converting to DataFrame
X_train = pd.DataFrame(X_train, columns=X_train.columns)
X_val = pd.DataFrame(X_val, columns=X_val.columns)

# Apply SMOTE to handle imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Scale the balanced training data and validation data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_val_scaled = scaler.transform(X_val)

# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train_balanced)

# Validation
val_preds = rf_model.predict(X_val_scaled)

# Confusion Matrix
cm = confusion_matrix(y_val, val_preds)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

# Calculate metrics
accuracy = accuracy_score(y_val, val_preds)
precision = precision_score(y_val, val_preds)
recall = recall_score(y_val, val_preds)       # TPR
f1 = f1_score(y_val, val_preds)
fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # Avoid division by zero

print("\nMetrics on Validation Set:")
print(f"Accuracy        : {accuracy:.4f}")
print(f"TPR (Recall)    : {recall:.4f}")
print(f"FPR             : {fpr:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"F1-score        : {f1:.4f}")

Confusion Matrix:
[[155141  24871]
 [ 46714  15488]]

Metrics on Validation Set:
Accuracy        : 0.7045
TPR (Recall)    : 0.2490
FPR             : 0.1382
Precision       : 0.3838
F1-score        : 0.3020


## XGBoost

In [5]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=1000, learning_rate=0.1, random_state=42)
xgb.fit(X_train_scaled, y_train_balanced)

val_preds = xgb.predict(X_val_scaled)
cm = confusion_matrix(y_val, val_preds)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

accuracy = accuracy_score(y_val, val_preds)
precision = precision_score(y_val, val_preds)
recall = recall_score(y_val, val_preds)       # TPR
f1 = f1_score(y_val, val_preds)
fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # Avoid division by zero

print("Metrics on Validation Set:")
print(f"Accuracy        : {accuracy:.4f}")
print(f"TPR (Recall)    : {recall:.4f}")
print(f"FPR             : {fpr:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"F1-score        : {f1:.4f}")

Confusion Matrix:
[[172992   7020]
 [ 56227   5975]]
Metrics on Validation Set:
Accuracy        : 0.7389
TPR (Recall)    : 0.0961
FPR             : 0.0390
Precision       : 0.4598
F1-score        : 0.1589


LightGBM


In [8]:
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Initialize LightGBM model
lgb_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=20,
    random_state=42
)

# Fit the model
lgb_model.fit(X_train_scaled, y_train_balanced)

# Validation
val_preds = lgb_model.predict(X_val_scaled)

# Confusion Matrix
cm = confusion_matrix(y_val, val_preds)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

# Calculate metrics
accuracy = accuracy_score(y_val, val_preds)
precision = precision_score(y_val, val_preds)
recall = recall_score(y_val, val_preds)
f1 = f1_score(y_val, val_preds)
fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # Avoid division by zero

print("\nMetrics on Validation Set:")
print(f"Accuracy        : {accuracy:.4f}")
print(f"TPR (Recall)    : {recall:.4f}")
print(f"FPR             : {fpr:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"F1-score        : {f1:.4f}")

[LightGBM] [Info] Number of positive: 721918, number of negative: 721918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032655 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2568
[LightGBM] [Info] Number of data points in the train set: 1443836, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\Andrew Hu\OneDrive\Documents\code\ML\myenv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Confusion Matrix:
[[170926   9086]
 [ 54898   7304]]

Metrics on Validation Set:
Accuracy        : 0.7358
TPR (Recall)    : 0.1174
FPR             : 0.0505
Precision       : 0.4456
F1-score        : 0.1859


In [9]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import joblib

# Create a voting classifier ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),  # Random Forest model
        ('xgb', xgb),      # XGBoost model
        ('lgb', lgb_model) # LightGBM model
    ],
    voting='soft',  # Use probability-based voting
    n_jobs=-1
)

# Fit the ensemble model
ensemble_model.fit(X_train_scaled, y_train_balanced)

# Validation
ensemble_preds = ensemble_model.predict(X_val_scaled)

# Confusion Matrix
cm = confusion_matrix(y_val, ensemble_preds)
tn, fp, fn, tp = cm.ravel()

print("\nEnsemble Model Confusion Matrix:")
print(cm)

# Calculate metrics
accuracy = accuracy_score(y_val, ensemble_preds)
precision = precision_score(y_val, ensemble_preds)
recall = recall_score(y_val, ensemble_preds)
f1 = f1_score(y_val, ensemble_preds)
fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # Avoid division by zero

print("\nMetrics on Validation Set:")
print(f"Accuracy        : {accuracy:.4f}")
print(f"TPR (Recall)    : {recall:.4f}")
print(f"FPR             : {fpr:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"F1-score        : {f1:.4f}")

# Save the ensemble model
joblib.dump(ensemble_model, 'ensemble_model.joblib')
print("\nEnsemble model saved as 'ensemble_model.joblib'")

c:\Users\Andrew Hu\OneDrive\Documents\code\ML\myenv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Ensemble Model Confusion Matrix:
[[173796   6216]
 [ 56427   5775]]

Metrics on Validation Set:
Accuracy        : 0.7414
TPR (Recall)    : 0.0928
FPR             : 0.0345
Precision       : 0.4816
F1-score        : 0.1557

Ensemble model saved as 'ensemble_model.joblib'


In [10]:
import joblib
# save rf model
joblib.dump(rf_model, 'rf_model.joblib')

['rf_model.joblib']